<a href="https://colab.research.google.com/github/vm7151810/Plant-Disease-Detection/blob/main/PlantDiseaseClassification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
#install the required dependencies
!pip install datasets evaluate matplotlib
!pip install torch torchvision transformers
!pip install accelerate -U
!pip install evaluate
!pip install wandb

In [ ]:
#import the modules
from datasets import load_dataset
import PIL
from datasets import load_dataset,concatenate_datasets
import evaluate
import matplotlib.pyplot as plt
from transformers import AutoImageProcessor
import warnings
warnings.filterwarnings('ignore') #ignore the warnings

In [ ]:
#mount the drive which has the dataset
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
path_to_folder = "/content/drive/MyDrive/PlantDiseaseDataset" #path to the dataset folder - https://www.kaggle.com/datasets/vipoooool/new-plant-diseases-dataset
dataset = load_dataset("imagefolder", data_dir=path_to_folder,drop_labels=False)

In [ ]:
from datasets import load_metric,list_metrics
metrics_list = list_metrics()
metric = load_metric("accuracy") #load the accuracy metric from the datset module

In [ ]:
labels = dataset["train"].features["label"].names
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = i
    id2label[i] = label
#mapping the labels to ID(integer)

In [ ]:
model_checkpoint = "google/vit-base-patch16-224-in21k" #using ViT, a pre-trained model, trained on ImageNet-21k dataset
image_processor  = AutoImageProcessor.from_pretrained(model_checkpoint) # config for preprocessing images

In [ ]:
# Pre-processing the dataset
from torchvision.transforms import (
    CenterCrop,
    Compose,
    Normalize,
    RandomHorizontalFlip,
    RandomVerticalFlip,
    RandomResizedCrop,
    Resize,
    ToTensor,
)

normalize = Normalize(mean=image_processor.image_mean, std=image_processor.image_std) #Normalizing the pixels about the mean
if "height" in image_processor.size:
    size = (image_processor.size["height"], image_processor.size["width"])
    crop_size = size
    max_size = None
elif "shortest_edge" in image_processor.size:
    size = image_processor.size["shortest_edge"]
    crop_size = (size, size)
    max_size = image_processor.size.get("longest_edge")

train_transforms = Compose(
        [
            RandomResizedCrop(crop_size),
            RandomHorizontalFlip(),
            RandomVerticalFlip(),
            ToTensor(),
            normalize,
        ]
    )
train_transforms_without_tensor = Compose(
        [
            RandomResizedCrop(crop_size),
            RandomHorizontalFlip(),
            RandomVerticalFlip(),
        ]
    )
train_transforms_keep_original = Compose(
        [
            Resize(crop_size),
            ToTensor(),
            normalize,
        ]
    )
train_transforms_keep_original_without_tensor = Compose(
        [
            Resize(crop_size),
        ]
    )
val_transforms = Compose(
        [
            Resize(size),
            CenterCrop(crop_size),
            ToTensor(),
            normalize,
        ]
    )

def preprocess_train(example_batch):
    """Apply train_transforms across a batch."""
    example_batch["pixel_values"] = [
        train_transforms(image.convert("RGB")) for image in example_batch["image"]
    ]
    example_batch["image_transform"] = [
        train_transforms_without_tensor(image.convert("RGB")) for image in example_batch["image"]
    ]
    return example_batch

def preprocess_original(example_batch):
    """Apply train_transforms across a batch."""
    example_batch["pixel_values"] = [
        train_transforms_keep_original(image.convert("RGB")) for image in example_batch["image"]
    ]
    example_batch["image_transform"] = [
        train_transforms_keep_original_without_tensor(image.convert("RGB")) for image in example_batch["image"]
    ]
    return example_batch

def preprocess_val(example_batch):
    """Apply val_transforms across a batch."""
    example_batch["pixel_values"] = [val_transforms(image.convert("RGB")) for image in example_batch["image"]]
    return example_batch

In [ ]:
dataset_train = dataset["train"]

In [ ]:
# Concatenating the dataset(with randomness) to increase the dataset for training
dataset_list = []
for i in range (16):
    dataset_list.append(dataset_train)
dataset_con = concatenate_datasets(dataset_list)

In [ ]:
#split in train and test dataset
splits = dataset_con.train_test_split(test_size=0.1)
train_ds = splits['train']
val_ds = splits['test']

In [ ]:
#let us see the class balance now
def showDistribution():
    distribution = evaluate.load("label_distribution")
    fig, ax = plt.subplots(1, 4, sharex=True, sharey=True, figsize=(15,5))
    ax[0].set_title("Original dataset - " + str(len(dataset_train)))
    results = distribution.compute(data=dataset_train['label'])
    ax[0].bar(results['label_distribution']['labels'], results['label_distribution']['fractions'])

    ax[1].set_title("Total concanated dataset " + str(len(dataset_con)))
    results = distribution.compute(data=dataset_con['label'])
    ax[1].bar(results['label_distribution']['labels'], results['label_distribution']['fractions'])

    ax[2].set_title(" Total  Training dataset " + str(len(train_ds)))
    results = distribution.compute(data=train_ds['label'])
    ax[2].bar(results['label_distribution']['labels'], results['label_distribution']['fractions'])

    ax[3].set_title(" Total validation dataset " + str(len(val_ds)))
    results = distribution.compute(data=val_ds['label'])
    ax[3].bar(results['label_distribution']['labels'], results['label_distribution']['fractions'])
    plt.show()
showDistribution()

In [ ]:
#we will set the transformation method to the list.
train_ds.set_transform(preprocess_train)
val_ds.set_transform(preprocess_val)

In [ ]:
# plot samples
import random
num = int(random.random() * 10000)
print(num)
def showimages(dataset,numberofimg):
    sorted_ds = dataset.sort('label')
    samples = sorted_ds.select(range(num, num+numberofimg))
    pointer = 0
    fig, ax = plt.subplots(5, 3, sharex=True, sharey=True, figsize=(10,6))
    for i in range(5):
        for j in range(3):
            ax[i,j].imshow(samples[pointer]['image_transform'])
            ax[i,j].set_title(f"{labels[samples[pointer]['label']]}")
            ax[i,j].axis('off')
            pointer+=1
    plt.show()
showimages(train_ds,15)


In [ ]:
from transformers import AutoModelForImageClassification, TrainingArguments, Trainer

model = AutoModelForImageClassification.from_pretrained(
    model_checkpoint,
    label2id=label2id,
    id2label=id2label, # aligning model's output to class labels in the dataset
    ignore_mismatched_sizes = True, # provide this in case you're planning to fine-tune an already fine-tuned checkpoint
)

In [ ]:
#Use GPU if available
import torch
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
#cuda.empty_cache()
print(device)
model.to(device)


In [ ]:
batch_size = 32
args = TrainingArguments(
    remove_unused_columns=False,
    evaluation_strategy="epoch", #evaluate at each epoch
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size,
    gradient_accumulation_steps=4, #gradients over 4 steps before updating the model's weights
    per_device_eval_batch_size=batch_size,
    num_train_epochs=30,
    warmup_ratio=0.1, #fraction of total steps to use for learning rate warmup
    logging_steps=10, #Logs training progress every 10 steps
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    push_to_hub=False,
    output_dir='./checkpoints', #Directory for saving files, but since wandb is used this is irrelavent
    report_to="wandb",
    resume_from_checkpoint=True,
    overwrite_output_dir=True,
    run_name="FaceRec_Training_Run", # to avoid warning
)

In [ ]:
import numpy as np

# the compute_metrics function takes a Named Tuple as input:
# predictions, which are the logits of the model as Numpy arrays,
# and label_ids, which are the ground-truth labels as Numpy arrays.
def compute_metrics(eval_pred):
    """Computes accuracy on a batch of predictions"""
    predictions = np.argmax(eval_pred.predictions, axis=1)
    return metric.compute(predictions=predictions, references=eval_pred.label_ids)

In [ ]:
import torch

def collate_fn(examples):
    pixel_values = torch.stack([example["pixel_values"] for example in examples])
    labels = torch.tensor([example["label"] for example in examples])
    return {"pixel_values": pixel_values, "labels": labels}

In [ ]:
!wandb login --relogin #force re-login to wandb

In [ ]:
import wandb
#import os

run = wandb.init(project="FaceRec", entity="qwertyytrewq34566543-bits-pilani") # Starting a new run in a FaceRec

In [ ]:
trainer = Trainer(
    model,
    args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=image_processor,
    compute_metrics=compute_metrics,
    data_collator=collate_fn
)

In [ ]:
#loading a saved model
run_path = "" # run path
model_path = wandb.restore('model.pth', run_path=run_path).name #Finding the path to the saved run
model.load_state_dict(torch.load(model_path)) # Loading the saved model
model.to(device) #use gpu if available

In [ ]:
train_results = trainer.train()

In [ ]:
#Saving the model
torch.save(model.state_dict(), './model.pth')
wandb.save('./model.pth')

In [ ]:
run.finish() # End the run

In [ ]:
metrics = trainer.evaluate()
# some nice to haves:
trainer.log_metrics("eval", metrics)
trainer.save_metrics("eval", metrics)